In [6]:
from langgraph.graph import StateGraph, START , END 
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage , AIMessage , AnyMessage
from typing import TypedDict , Literal  , Annotated
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt , Command
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
#from langchain_nvidia_ai_endpoints import ChatNVIDIA

#print(ChatNVIDIA.get_available_models())

In [8]:
llm = ChatNVIDIA(model="nvidia/nemotron-mini-4b-instruct", temperature=0)

In [9]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

Developing Chat Node (HITL)

In [10]:
def chat_node(state: ChatState):

    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })
    
    if decision["approved"] == 'no':
        return {"messages": [AIMessage(content="Not approved.")]}

    else:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

In [11]:
builder = StateGraph(ChatState)

builder.add_node("chat", chat_node)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Checkpointer is required for interrupts
checkpointer = MemorySaver()


app = builder.compile(checkpointer=checkpointer)

This chat node is basically a HITL node 

In [17]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": '1234'}}

initial_input = {
    "messages": [
        ("user", "Explain what is computer in very simple terms.")
    ]
}

# Invoke the graph 
result = app.invoke(initial_input, config=config)
result


{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='9a875a8c-d562-4b3b-9a3b-a177c3e852af'),
  AIMessage(content="Sure, I'd be happy to explain gradient descent in simple terms!\n\nGradient descent is a fundamental optimization algorithm used to find the minimum value of a function, often in machine learning and data analysis. Here's a simple breakdown:\n\n1. **Function to Minimize**: We start with a function that we want to optimize, like the cost function in a machine learning model. This function takes in some input (weights or parameters) and returns a value (cost or loss). For example, in a linear regression model, the cost function might be the sum of the squared differences between the predicted and actual values.\n\n2. **Initial Point**: We start with an initial point (or set of points) for our optimization. This could be random, or we might have some prior knowledge about the optimal solution.\n\n3

In [18]:
message = result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'Explain what is computer in very simple terms.',
 'instruction': 'Approve this question? yes/no'}

In [19]:
user_input = input(f"\nBackend message = {message} \n Approve this question (y/n): ")

In [20]:
final_result = app.invoke(
    Command(resume = {"approved" : user_input}),
    config = config
)

In [21]:
print(final_result)

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='9a875a8c-d562-4b3b-9a3b-a177c3e852af'), AIMessage(content="Sure, I'd be happy to explain gradient descent in simple terms!\n\nGradient descent is a fundamental optimization algorithm used to find the minimum value of a function, often in machine learning and data analysis. Here's a simple breakdown:\n\n1. **Function to Minimize**: We start with a function that we want to optimize, like the cost function in a machine learning model. This function takes in some input (weights or parameters) and returns a value (cost or loss). For example, in a linear regression model, the cost function might be the sum of the squared differences between the predicted and actual values.\n\n2. **Initial Point**: We start with an initial point (or set of points) for our optimization. This could be random, or we might have some prior knowledge about the optimal solution.\n\n3. 